# TRAINING TWO RECURRENT RNN MODELS WITHOUT FINE-TUNING

## SETTING UP THE ENVIRONMENT

In [1]:
!pip install -q evaluate seqeval transformers datasets==3.6.0 huggingface_hub pytorch-crf conllu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00


Importing the libraries:

In [ ]:
import os
import math
import spacy
import evaluate

import numpy as np

from huggingface_hub import login
from collections     import Counter
from copy            import deepcopy
from spacy           import displacy
from datasets        import Value       , \
                            Sequence    , \
                            Features    , \
                            ClassLabel  , \
                            DatasetDict , \
                            load_dataset, \
                            concatenate_datasets

from torchcrf import CRF

import torch
import torch.nn.functional as F
import torch.nn            as nn

from torch.utils.data   import Dataset   , \
                               DataLoader
from torch.nn.utils.rnn import pad_sequence        , \
                               pack_padded_sequence, \
                               pad_packed_sequence

Authenticating in hugging face:

## LOAD THE DATASET

Load the dataset:

In [ ]:
sources = {
    "lener_br"    : ("lfcc/portuguese_ner", None),
    "nerde"       : ("Gpaiva/NERDE"       , None),
    "harem"       : ("Linguateca/harem"   , "selective"),
    "wikiann_pt"  : ("wikiann"                   , "pt"       ),
    "uner_pt"     : ("universalner/universal_ner", "pt_bosque"),
}

datasets = {name : load_dataset(repo  ,
                                subset,
                                trust_remote_code=True) \
                   if   subset
                   else load_dataset(repo)
            for name, (repo, subset) in sources.items()}
datasets

## PREPROCESS THE DATA

In [5]:
features   = datasets["wikiann_pt"]["train"   ].features
label_list = features["ner_tags"              ].feature.names

num_labels = len(label_list)

id2label = {i     : label for i, label in enumerate(label_list)}
label2id = {label : i     for i, label in enumerate(label_list)}

print("Labels:", ", ".join(label_list))

Labels: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC


In [6]:
def map_label(original):
    o = original.upper()

    if "PESSOA" in o or \
       "PER"    in o:
        if o.startswith("I-"):
            return "I-PER"

        return "B-PER"

    if "ORG"         in o or \
       "ORGANIZACAO" in o or \
       "ORGANIZAÇÃO" in o:
        if o.startswith("I-"):
            return "I-ORG"

        return "B-ORG"

    if "LOC"   in o or \
       "LOCAL" in o:
        if o.startswith("I-"):
            return "I-LOC"

        return "B-LOC"

    return "O"


def normalize_dataset(ds):
    train_features = ds["train"].features

    def convert(example):
        original_tags = example["ner_tags"]

        if isinstance(train_features["ner_tags"].feature, ClassLabel):
            names           = train_features["ner_tags"].feature.names
            original_labels = [names[t] for t in original_tags]
        else:
            original_labels = original_tags

        new_tags = [label2id[map_label(lbl)] for lbl in original_labels]

        return {
            "tokens"   : example["tokens"],
            "ner_tags" : new_tags
        }

    new_splits = {}
    for split in ["train", "validation", "test"]:
        if split in ds:
            new_splits[split] = ds[split].map(
                convert,
                remove_columns=ds[split].column_names
            )

    features = Features({
        "tokens"   : Sequence(Value("string")),
        "ner_tags" : Sequence(Value("int64" ))
    })

    for split in new_splits:
        new_splits[split] = new_splits[split].cast(features)

    return DatasetDict(new_splits)


normalized = []
for name, ds in datasets.items():
    print()
    print(f"Normalizing {name}...")

    normalized.append(normalize_dataset(ds))


def concat(split):
    parts = [
        ds[split]
        for ds in normalized
        if split in ds
    ]

    return concatenate_datasets(parts)


print()
print("Concatenating...")

train = concat("train"     )
val   = concat("validation")
test  = concat("test"      )

dataset = DatasetDict({
    "train"      : train,
    "validation" : val  ,
    "test"       : test ,
})


print()
print("Dataset:")
print(dataset)


Normalizing lener_br...


Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3716 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/930 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1167 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/7018 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1172 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1167 [00:00<?, ? examples/s]


Concatenating...

Dataset:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 39499
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 12654
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 13914
    })
})


Obtaining the vocabulary:

In [7]:
PAD_TOKEN = "[PAD]"
UNK_TOKEN = "[UNK]"

counter = Counter()
for ex in dataset["train"]:
    counter.update(ex["tokens"])

word2idx = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
}
for word, _ in counter.most_common():
    word2idx.setdefault(word, len(word2idx))

idx2word = {i: w for w, i in word2idx.items()}

vocab_size = len(word2idx)

print("Vocabulary size:", vocab_size)

Vocabulary size: 66192


Encode token sequences into index IDs and align them with their corresponding NER labels:

In [8]:
def encode_example(example):
    unk_idx   = word2idx[UNK_TOKEN]

    input_ids = [
        word2idx.get(tok, unk_idx)
        for tok in example["tokens"]
    ]

    return {
        "input_ids" : input_ids,
        "labels"    : example["ner_tags"],
    }

encoded_dataset = dataset.map(
    encode_example,
    remove_columns=["tokens", "ner_tags"]
)

example = dataset        ["train"][0]
encoded = encoded_dataset["train"][0]

for tok, idx, label in zip(example["tokens"   ], \
                           encoded["input_ids"], \
                           encoded["labels"   ]):
    print(f"{tok:10} ({idx:5}) - {id2label[label]}")

Map:   0%|          | 0/39499 [00:00<?, ? examples/s]

Map:   0%|          | 0/12654 [00:00<?, ? examples/s]

Map:   0%|          | 0/13914 [00:00<?, ? examples/s]

Filiação   (   64) - O
:          (   16) - O
Antonio    (  181) - B-PER
Joaquim    (  693) - I-PER
Aguiar     ( 2534) - I-PER
e          (    7) - O
Engracia   (29214) - B-PER
Maria      (   48) - I-PER
.          (    4) - O
Natural    (   62) - O
e/ou       (   57) - O
residente  (   63) - O
em         (   10) - O
CUNHA      (29215) - B-LOC
,          (    2) - O
Santa      (  115) - B-LOC
Maria      (   48) - I-LOC
,          (    2) - O
actual     (   61) - O
concelho   (   58) - O
de         (    3) - O
PAREDES    ( 3678) - B-LOC
COURA      ( 4674) - I-LOC
e          (    7) - O
distrito   (   54) - O
(          (    9) - O
ou         (   30) - O
país       (   52) - O
)          (    8) - O
Viana      (  505) - B-LOC
do         (   13) - I-LOC
Castelo    (  359) - I-LOC
.          (    4) - O


Previewing examples from the custom NERDataset:

In [9]:
class NERDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]

        return torch.tensor(example["input_ids"], dtype=torch.long), \
               torch.tensor(example["labels"   ], dtype=torch.long)


train_ds = NERDataset(encoded_dataset["train"     ])
val_ds   = NERDataset(encoded_dataset["validation"])
test_ds  = NERDataset(encoded_dataset["test"      ])

input_ids, labels = train_ds[0]
print("Input  =", input_ids)
print("Labels =", labels   )

Input  = tensor([   64,    16,   181,   693,  2534,     7, 29214,    48,     4,    62,
           57,    63,    10, 29215,     2,   115,    48,     2,    61,    58,
            3,  3678,  4674,     7,    54,     9,    30,    52,     8,   505,
           13,   359,     4])
Labels = tensor([0, 0, 1, 2, 2, 0, 1, 2, 0, 0, 0, 0, 0, 5, 0, 5, 6, 0, 0, 0, 0, 5, 6, 0,
        0, 0, 0, 0, 0, 5, 6, 6, 0])


Custom collate function for padding variable-length NER sequences:

In [10]:
def collate_fn(batch):
    """
    batch: [(input_ids, labels),...]
      - input_ids_padded: [batch, max_len]
      - labels_padded   : [batch, max_len]
      - lengths         : [batch]
    """
    pad_idx = word2idx[PAD_TOKEN]

    filtered_batch = []
    removed_idx    = []

    for i, (input_ids, labels) in enumerate(batch):
        if len(input_ids) > 0:
            filtered_batch.append((input_ids, labels))
        else:
            removed_idx   .append(i)

    input_seqs, label_seqs = zip(*filtered_batch)

    input_padded = pad_sequence(
        input_seqs,
        batch_first  =True   ,
        padding_value=pad_idx,
    )
    labels_padded = pad_sequence(
        label_seqs,
        batch_first  =True   ,
        padding_value=pad_idx,
    )

    return input_padded , \
           labels_padded, \
           torch.tensor([len(seq) for seq in input_seqs], dtype=torch.long)

batch_size = 16

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True , collate_fn=collate_fn)
val_loader   = DataLoader(val_ds  , batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds , batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [11]:
example_batch = next(iter(train_loader))

input_padded, labels_padded, lengths = example_batch

print("Input  padded shape:", input_padded .shape)
print("Labels padded shape:", labels_padded.shape)
print("Lengths:", lengths.tolist())

idx = 0

print()
print("First example in batch:")
print("Tokens:", " ".join([idx2word  [i.item()] for i in input_padded [idx][:lengths[idx]]]))
print("Labels:", " ".join([label_list[l.item()] for l in labels_padded[idx][:lengths[idx]]]))

Input  padded shape: torch.Size([16, 68])
Labels padded shape: torch.Size([16, 68])
Lengths: [31, 15, 11, 63, 24, 27, 3, 3, 1, 7, 7, 14, 68, 34, 6, 3]

First example in batch:
Tokens: Faixas de porcentagem da representatividade dos tipos de peças ( caldeiradas , usinadas e revendidas ) sobre o total de vendas no mercado de peças de reposição para colhedoras de cana-de-açúcar
Labels: O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O


## AUXILIARY FUNCTIONS

Auxiliary functions:

In [12]:
seqeval = evaluate.load("seqeval")


def evaluate_model(model, data_loader):
    model.eval()

    all_pred_labels = []
    all_true_labels = []

    with torch.no_grad():
        for input_ids, labels, lengths in data_loader:
            input_ids = input_ids.to(device)
            labels    = labels   .to(device)
            lengths   = lengths  .to(device)

            logits = model(input_ids, lengths)
            preds  = logits.argmax(dim=-1)

            preds  = preds .cpu().numpy()
            labels = labels.cpu().numpy()

            for p_seq, l_seq in zip(preds, labels):
                p_labels = []
                l_labels = []

                for p, l in zip(p_seq, l_seq):
                    p_labels.append(label_list[p])
                    l_labels.append(label_list[l])

                all_pred_labels.append(p_labels)
                all_true_labels.append(l_labels)

    return seqeval.compute(
        predictions=all_pred_labels,
        references =all_true_labels,
    )


def evaluate_model_crf(model, data_loader):
    model.eval()

    all_pred_labels = []
    all_true_labels = []

    with torch.no_grad():
        for input_ids, labels, lengths in data_loader:
            input_ids = input_ids.to(device)
            labels    = labels   .to(device)
            lengths   = lengths  .to(device)

            preds      = model(input_ids, lengths, labels=None)
            labels_np  = labels .cpu().numpy()
            lengths_np = lengths.cpu().numpy()

            for p_seq, l_seq, L in zip(preds     , \
                                       labels_np , \
                                       lengths_np):
                p_seq = p_seq[:L]
                l_seq = l_seq[:L]

                all_pred_labels.append([label_list[int(p)] for p in p_seq])
                all_true_labels.append([label_list[int(l)] for l in l_seq])

    return seqeval.compute(
        predictions=all_pred_labels,
        references =all_true_labels,
    )


def predict_ner(model, tokens):
    unk_idx = word2idx[UNK_TOKEN]

    model.eval()

    input_ids = [
        word2idx.get(tok, unk_idx)
        for tok in tokens
    ]
    input_tensor = torch.tensor(input_ids    , dtype=torch.long).unsqueeze(0).to(device)
    lengths      = torch.tensor([len(tokens)], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_tensor, lengths)
        preds  = logits.argmax(dim=-1).squeeze(0).cpu().numpy()

    results = []
    for tok, p in zip(tokens, preds):
        results.append((tok, id2label[int(p)]))

    return results


def predict_ner_crf(model, tokens):
    unk_idx = word2idx[UNK_TOKEN]

    model.eval()

    input_ids = [
        word2idx.get(tok, unk_idx)
        for tok in tokens
    ]
    input_tensor = torch.tensor(input_ids    , dtype=torch.long).unsqueeze(0).to(device)
    lengths      = torch.tensor([len(tokens)], dtype=torch.long).to(device)

    with torch.no_grad():
        preds = model(
            input_tensor,
            lengths     ,
            labels=None
        )[0]

    results = []
    for tok, p in zip(tokens, preds):
        results.append((tok, id2label[int(p)]))

    return results


def displacy_from_predict(model, tokens, fn_predict):
    predictions = fn_predict(model, tokens)

    text = " ".join(tokens)

    ents    = []
    current = None
    offset  = 0

    for token, tag in predictions:
        token_start = text.find(token, offset)
        token_end   = token_start + len(token)

        if tag.startswith("B-"):
            if current:
                ents.append(current)

            current = {
                "start" : token_start,
                "end"   : token_end  ,
                "label" : tag[2:]
            }
        elif tag.startswith("I-") and current and current["label"] == tag[2:]:
            current["end"] = token_end
        else:
            if current:
                ents.append(current)
                current = None

        offset = token_end

    if current:
        ents.append(current)

    docs = [{
        "text"  : text,
        "ents"  : ents,
    }]

    colors = {
        "PER" : "linear-gradient(90deg, #999, #ccc)"      ,
        "LOC" : "linear-gradient(90deg, #aa9cfc, #fc9ce7)",
        "ORG" : "linear-gradient(90deg, #ffcc70, #ff9a3c)",
    }

    options = {"ents": ["PER", "LOC", "ORG"], "colors": colors}

    return docs, options


def count_parameters(model):
    return sum(p.numel() for p in model.parameters()), \
           sum(p.numel() for p in model.parameters() if p.requires_grad)

Parameters for models:

In [13]:
device = torch.device("cuda"
                      if torch.cuda.is_available()
                      else "cpu")
print("Device:", device)

embedding_dim = 128
hidden_dim    = 128

Device: cuda


## BILSTM WITH LINEAR TOKEN CLASSIFICATION

**BiLSTM base**: embedding → packed bidirectional LSTM → linear classifier for token-level NER logits.

In [14]:
class BiLSTMTagger(nn.Module):
    def __init__(
        self         ,
        vocab_size   ,
        embedding_dim,
        hidden_dim   ,
        num_labels   ,
        pad_idx      ,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size   ,
            embedding_dim =embedding_dim,
            padding_idx   =pad_idx
        )

        self.lstm = nn.LSTM(
            input_size =embedding_dim,
            hidden_size=hidden_dim   ,
            num_layers   =1   ,
            batch_first  =True,
            bidirectional=True,
        )

        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)

        packed = pack_padded_sequence(
            embedded     ,
            lengths.cpu(),
            batch_first   =True ,
            enforce_sorted=False,
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        return self.fc(output)

Before starting the training process, we define two fundamental components:

* **Loss function (`criterion`)**: We use `CrossEntropyLoss` because the problem is a *multi-class classification task at the token level*.

* **Optimizer (`optimizer`)**: We use `AdamW`, a stable and modern variant of Adam that provides appropriate regularization and efficient convergence.

In [15]:
model_small = BiLSTMTagger(
    vocab_size   =vocab_size   ,
    embedding_dim=embedding_dim,
    hidden_dim   =hidden_dim   ,
    num_labels   =num_labels   ,
    pad_idx      =word2idx[PAD_TOKEN]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW  (model_small.parameters(), lr=5e-4)

In [ ]:
total, trainable = count_parameters(model_small)

print(f"Total parameters       : {total    :,}")
print(f"Trainable parameters   : {trainable:,}")

Total parameters       : 8,738,567
Trainable parameters   : 8,738,567


In [ ]:
tokens = ["O", "presidente", "Luiz", "Inácio", "Lula", "da", "Silva", "visitou", "o", "Ministério", "Público", "de", "São", "Paulo", "."]

num_epochs = 50
patience   =  5
best_val_loss     = float("inf")
epochs_no_improve = 0

os.makedirs("models", exist_ok=True)

for epoch in range(1, num_epochs + 1):
    model_small.train()

    total_loss = 0.0
    for input_ids, labels, lengths in train_loader:
        input_ids = input_ids.to(device)
        labels    = labels   .to(device)
        lengths   = lengths  .to(device)

        optimizer.zero_grad()

        logits = model_small(input_ids, lengths)
        loss   = criterion(
            logits.view(-1, num_labels),
            labels.view(-1)
        )

        loss .backward()
        optimizer.step()

        total_loss += loss.item()

    model_small.eval()

    val_loss = 0.0
    with torch.no_grad():
        for input_ids, labels, lengths in val_loader:
            input_ids = input_ids.to(device)
            labels    = labels   .to(device)
            lengths   = lengths  .to(device)

            logits    = model_small(input_ids, lengths)
            val_loss += criterion  (
                logits.view(-1, num_labels),
                labels.view(-1)
            ).item()

    avg_loss     = total_loss / len(train_loader)
    avg_val_loss = val_loss   / len(val_loader  )

    print()
    print(f"Epoch {epoch}/{num_epochs}")
    print(f"  Train Loss : {avg_loss    :.4f}")
    print(f"  Val Loss   : {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss     = avg_val_loss
        epochs_no_improve = 0

        best_model_wts = deepcopy(model_small.state_dict())
        torch.save(
            best_model_wts,
            os.path.join("models", "model_small.pt")
        )

        print(f"  🔹 Best model saved ({epoch} epoch)!")
    else:
        epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print()
            print(f"⚠️ Early stopping triggered! No improvement for {patience} epochs.")
            break

    model_small.eval()
    with torch.no_grad():
        docs, options = displacy_from_predict(model_small, tokens, predict_ner)
        displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)


Epoch 1/50
  Train Loss : 0.9347
  Val Loss   : 0.5365
  🔹 Best model saved (1 epoch)!



Epoch 2/50
  Train Loss : 0.2769
  Val Loss   : 0.2692
  🔹 Best model saved (2 epoch)!


Evaluating the model:

In [ ]:
small_model_path = os.path.join("models", "model_small.pt")

model_small.load_state_dict(
    torch.load(
        small_model_path,
        map_location=device
    )
)
model_small.to  (device)
model_small.eval()

test_metrics = evaluate_model(model_small, test_loader)

print("=== Test Set Results ===")
print(f"  F1 Score   : {test_metrics['overall_f1'       ]:.4f}")
print(f"  Precision  : {test_metrics['overall_precision']:.4f}")
print(f"  Recall     : {test_metrics['overall_recall'   ]:.4f}")
print(f"  Accuracy   : {test_metrics['overall_accuracy' ]:.4f}")

In [ ]:
test_texts = [
    "João encontrou Maria ontem à noite."     ,
    "Estou indo para João Pessoa amanhã cedo.",
    "A Google lançou um novo modelo de IA."   ,
    "A Universidade Federal da Paraíba convidou Ana Beatriz para apresentar sua pesquisa em São Paulo."                 ,
    "O presidente da Microsoft Brasil, André Oliveira, visitou o escritório em Fortaleza para anunciar novas parcerias.",
    "Mariana trabalhou três anos na IBM, antes de se mudar para o Rio de Janeiro para atuar no BNDES."                  ,
    "Em 2024, Carlos Eduardo foi contratado pelo Banco do Brasil após concluir seu mestrado na USP, em São Paulo."      ,
    "A Meta divulgou um relatório em que Sheryl Sandberg mencionou iniciativas de segurança digital nas operações da empresa."                ,
    "Durante a reunião em Brasília, representantes da ONU e do Ministério da Justiça discutiram estratégias para reduzir crimes cibernéticos.",
    "Pedro Henrique trabalhou por cinco anos na Petrobras no Rio de Janeiro, até receber uma proposta da Amazon em Seattle."                  ,
]

nlp = spacy.blank("pt")

for text in test_texts:
    docs, options = displacy_from_predict(model_small                ,
                                          [t.text for t in nlp(text)],
                                          predict_ner)
    displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

Releasing memory allocated for the model:

In [ ]:
del model_small

torch.cuda.empty_cache()

## BILSTM LARGE

**Enhanced BiLSTM Architecture**:

Below we define an improved BiLSTM model that introduces additional depth, dropout regularization, and an intermediate feed-forward layer to increase the network’s representational capacity.

In [ ]:
class BiLSTMTaggerLarge(nn.Module):
    def __init__(
        self         ,
        vocab_size   ,
        embedding_dim,
        hidden_dim   ,
        num_labels   ,
        pad_idx      ,
        dropout=0.2
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size   ,
            embedding_dim =embedding_dim,
            padding_idx   =pad_idx      ,
        )
        self.emb_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size =embedding_dim,
            hidden_size=hidden_dim   ,
            num_layers   =3   ,
            batch_first  =True,
            bidirectional=True,
            dropout=dropout
        )

        self.relu         = nn.ReLU()
        self.intermediate = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc           = nn.Linear(hidden_dim    , num_labels)

    def forward(self, input_ids, lengths):
        embedded = self.embedding  (input_ids)
        embedded = self.emb_dropout(embedded )

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first   =True ,
            enforce_sorted=False,
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        h      = self.intermediate(output)
        h      = self.relu(h)
        logits = self.fc  (h)

        return logits

In [ ]:
model_medium = BiLSTMTaggerLarge(
    vocab_size   =vocab_size   ,
    embedding_dim=embedding_dim,
    hidden_dim   =hidden_dim   ,
    num_labels   =num_labels   ,
    pad_idx      =word2idx[PAD_TOKEN]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW  (model_medium.parameters(), lr=5e-4)

In [ ]:
total, trainable = count_parameters(model_medium)

print(f"Total parameters       : {total    :,}")
print(f"Trainable parameters   : {trainable:,}")

In [ ]:
tokens = ["O", "presidente", "Luiz", "Inácio", "Lula", "da", "Silva", "visitou", "o", "Ministério", "Público", "de", "São", "Paulo", "."]

num_epochs = 50
patience   =  5
best_val_loss     = float("inf")
epochs_no_improve = 0

os.makedirs("models", exist_ok=True)

for epoch in range(1, num_epochs + 1):
    model_medium.train()

    total_loss = 0.0
    for input_ids, labels, lengths in train_loader:
        input_ids = input_ids.to(device)
        labels    = labels   .to(device)
        lengths   = lengths  .to(device)

        optimizer.zero_grad()

        logits = model_medium(input_ids, lengths)
        loss   = criterion(
            logits.view(-1, num_labels),
            labels.view(-1)
        )

        loss .backward()
        optimizer.step()

        total_loss += loss.item()

    model_medium.eval()

    val_loss = 0.0
    with torch.no_grad():
        for input_ids, labels, lengths in val_loader:
            input_ids = input_ids.to(device)
            labels    = labels   .to(device)
            lengths   = lengths  .to(device)

            logits    = model_medium(input_ids, lengths)
            val_loss += criterion  (
                logits.view(-1, num_labels),
                labels.view(-1)
            ).item()

    avg_loss     = total_loss / len(train_loader)
    avg_val_loss = val_loss   / len(val_loader  )

    print()
    print(f"Epoch {epoch}/{num_epochs}")
    print(f"  Train Loss : {avg_loss    :.4f}")
    print(f"  Val Loss   : {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss     = avg_val_loss
        epochs_no_improve = 0

        best_model_wts = deepcopy(model_medium.state_dict())
        torch.save(
            best_model_wts,
            os.path.join("models", "model_medium.pt")
        )

        print(f"  🔹 Best model saved ({epoch} epoch)!")
    else:
        epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print()
            print(f"⚠️ Early stopping triggered! No improvement for {patience} epochs.")
            break

    model_medium.eval()
    with torch.no_grad():
        docs, options = displacy_from_predict(model_medium, tokens, predict_ner)
        displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

New evaluations for comparison:

In [ ]:
medium_model_path = os.path.join("models", "model_medium.pt")

model_medium.load_state_dict(
    torch.load(
        medium_model_path,
        map_location=device
    )
)
model_medium.to  (device)
model_medium.eval()

test_metrics = evaluate_model(model_medium, test_loader)

print("=== Test Set Results ===")
print(f"  F1 Score   : {test_metrics['overall_f1'       ]:.4f}")
print(f"  Precision  : {test_metrics['overall_precision']:.4f}")
print(f"  Recall     : {test_metrics['overall_recall'   ]:.4f}")
print(f"  Accuracy   : {test_metrics['overall_accuracy' ]:.4f}")

In [ ]:
test_texts = [
    "João encontrou Maria ontem à noite."     ,
    "Estou indo para João Pessoa amanhã cedo.",
    "A Google lançou um novo modelo de IA."   ,
    "A Universidade Federal da Paraíba convidou Ana Beatriz para apresentar sua pesquisa em São Paulo."                 ,
    "O presidente da Microsoft Brasil, André Oliveira, visitou o escritório em Fortaleza para anunciar novas parcerias.",
    "Mariana trabalhou três anos na IBM, antes de se mudar para o Rio de Janeiro para atuar no BNDES."                  ,
    "Em 2024, Carlos Eduardo foi contratado pelo Banco do Brasil após concluir seu mestrado na USP, em São Paulo."      ,
    "A Meta divulgou um relatório em que Sheryl Sandberg mencionou iniciativas de segurança digital nas operações da empresa."                ,
    "Durante a reunião em Brasília, representantes da ONU e do Ministério da Justiça discutiram estratégias para reduzir crimes cibernéticos.",
    "Pedro Henrique trabalhou por cinco anos na Petrobras no Rio de Janeiro, até receber uma proposta da Amazon em Seattle."                  ,
]

nlp = spacy.blank("pt")

for text in test_texts:
    docs, options = displacy_from_predict(model_medium               ,
                                          [t.text for t in nlp(text)],
                                          predict_ner)
    displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
del model_medium

torch.cuda.empty_cache()

## BILSTM WITH CRF

In [ ]:
class BiLSTM_Attn_CRF_Tagger(nn.Module):
    def __init__(
        self         ,
        vocab_size   ,
        embedding_dim,
        hidden_dim   ,
        num_labels   ,
        pad_idx      ,
        num_layers=3  ,
        dropout   =0.3,
    ):
        super().__init__()

        self.num_labels = num_labels
        self.pad_idx    = pad_idx

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size   ,
            embedding_dim =embedding_dim,
            padding_idx   =pad_idx
        )
        self.emb_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size   =embedding_dim,
            hidden_size  =hidden_dim   ,
            num_layers   =num_layers   ,
            batch_first  =True         ,
            bidirectional=True         ,
            dropout      =dropout
        )

        self.attn         = nn.Linear   (hidden_dim * 2, hidden_dim * 2)
        self.attn_vector  = nn.Parameter(torch.randn(hidden_dim * 2   ))
        self.attn_dropout = nn.Dropout  (dropout)

        self.fc  = nn.Linear(hidden_dim * 4, num_labels)
        self.crf = CRF      (num_labels    , batch_first=True)

    def forward(self, input_ids, lengths, labels=None):
        mask = (input_ids != self.pad_idx)

        embedded = self.embedding  (input_ids)
        embedded = self.emb_dropout(embedded )

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first   =True ,
            enforce_sorted=False,
        )
        packed_output, _ = self.lstm(packed)
        lstm_out     , _ = pad_packed_sequence(packed_output, batch_first=True)

        attn_scores = torch      .tanh       (self.attn(lstm_out))
        attn_scores = torch      .matmul     (attn_scores, self.attn_vector)
        attn_scores = attn_scores.masked_fill(~mask      , float("-inf")   )

        attn_weights     = torch       .softmax  (attn_scores, dim=-1)
        attn_weights_exp = attn_weights.unsqueeze(-1)

        context     = torch.sum(attn_weights_exp * lstm_out, dim=1)
        context_exp = context.unsqueeze(1).expand(-1, lstm_out.size(1), -1)

        combined = torch.cat         ([lstm_out, context_exp], dim=-1)
        combined = self .attn_dropout(combined)

        emissions = self.fc(combined)

        if labels is not None:
            return -self.crf(emissions, labels, mask=mask, reduction="mean")

        return self.crf.decode(emissions, mask=mask)

In [ ]:
model_large = BiLSTM_Attn_CRF_Tagger(
    vocab_size   =vocab_size   ,
    embedding_dim=embedding_dim,
    hidden_dim   =hidden_dim   ,
    num_labels   =num_labels   ,
    pad_idx      =word2idx[PAD_TOKEN],
).to(device)

optimizer = torch.optim.AdamW(model_large.parameters(), lr=5e-4)

In [ ]:
total, trainable = count_parameters(model_large)

print(f"Total parameters       : {total    :,}")
print(f"Trainable parameters   : {trainable:,}")

In [ ]:
tokens = ["O", "presidente", "Luiz", "Inácio", "Lula", "da", "Silva", "visitou", "o", "Ministério", "Público", "de", "São", "Paulo", "."]

num_epochs = 50
patience   =  5
best_val_loss     = float("inf")
epochs_no_improve = 0

os.makedirs("models", exist_ok=True)

for epoch in range(1, num_epochs + 1):
    model_large.train()

    total_loss = 0.0
    for input_ids, labels, lengths in train_loader:
        input_ids = input_ids.to(device)
        labels    = labels   .to(device)
        lengths   = lengths  .to(device)

        optimizer.zero_grad()

        loss = model_large(input_ids,
                           lengths  ,
                           labels=labels)

        loss .backward()
        optimizer.step()
        total_loss += loss.item()

    model_large.eval()

    val_loss = 0.0
    with torch.no_grad():
        for input_ids, labels, lengths in val_loader:
            input_ids = input_ids.to(device)
            labels    = labels   .to(device)
            lengths   = lengths  .to(device)

            val_loss += model_large(input_ids,
                                    lengths  ,
                                    labels=labels).item()

    avg_loss     = total_loss / len(train_loader)
    avg_val_loss = val_loss   / len(val_loader  )

    print()
    print(f"Epoch {epoch}/{num_epochs}")
    print(f"  Train Loss : {avg_loss    :.4f}")
    print(f"  Val Loss   : {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss     = avg_val_loss
        epochs_no_improve = 0

        best_model_wts = deepcopy(model_large.state_dict())
        torch.save(
            best_model_wts,
            os.path.join("models", "model_large.pt")
        )

        print(f"  🔹 Best model saved ({epoch} epoch)!")
    else:
        epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print()
            print(f"⚠️ Early stopping triggered! No improvement for {patience} epochs.")
            break

    model_large.eval()
    with torch.no_grad():
        docs, options = displacy_from_predict(model_large, tokens, predict_ner_crf)
        displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
large_model_path = os.path.join("models", "model_large.pt")

model_large.load_state_dict(
    torch.load(
        large_model_path,
        map_location=device
    )
)
model_large.to  (device)
model_large.eval()

test_metrics = evaluate_model_crf(model_large, test_loader)

print("=== Test Set Results ===")
print(f"  F1 Score   : {test_metrics['overall_f1'       ]:.4f}")
print(f"  Precision  : {test_metrics['overall_precision']:.4f}")
print(f"  Recall     : {test_metrics['overall_recall'   ]:.4f}")
print(f"  Accuracy   : {test_metrics['overall_accuracy' ]:.4f}")

In [ ]:
test_texts = [
    "João encontrou Maria ontem à noite."     ,
    "Estou indo para João Pessoa amanhã cedo.",
    "A Google lançou um novo modelo de IA."   ,
    "A Universidade Federal da Paraíba convidou Ana Beatriz para apresentar sua pesquisa em São Paulo."                 ,
    "O presidente da Microsoft Brasil, André Oliveira, visitou o escritório em Fortaleza para anunciar novas parcerias.",
    "Mariana trabalhou três anos na IBM, antes de se mudar para o Rio de Janeiro para atuar no BNDES."                  ,
    "Em 2024, Carlos Eduardo foi contratado pelo Banco do Brasil após concluir seu mestrado na USP, em São Paulo."      ,
    "A Meta divulgou um relatório em que Sheryl Sandberg mencionou iniciativas de segurança digital nas operações da empresa."                ,
    "Durante a reunião em Brasília, representantes da ONU e do Ministério da Justiça discutiram estratégias para reduzir crimes cibernéticos.",
    "Pedro Henrique trabalhou por cinco anos na Petrobras no Rio de Janeiro, até receber uma proposta da Amazon em Seattle."                  ,
]

nlp = spacy.blank("pt")

for text in test_texts:
    docs, options = displacy_from_predict(model_large                ,
                                          [t.text for t in nlp(text)],
                                          predict_ner_crf)
    displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
del model_large

torch.cuda.empty_cache()

## TRANSFORMERS FROM SCRATCH

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()

        self.d_model = d_model

        pe = self._generate_pe(max_len )
        self.register_buffer  ("pe", pe)

    def _generate_pe(self, max_len):
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self.d_model, 2).float() * (-math.log(10000.0) / self.d_model)
        )

        pe          = torch.zeros(max_len, self.d_model)
        pe[:, 0::2] = torch.sin  (position * div_term  )

        if self.d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0)

    def _extend_pe(self, new_len):
        current_len = self.pe.size(1)
        extra_len   = new_len - current_len

        pe_extra = self._generate_pe(extra_len).to(device)
        pe_new   = torch.cat([self.pe, pe_extra], dim=1)

        self.pe = pe_new

    def forward(self, x):
        seq_len = x.size(1)

        if seq_len > self.pe.size(1):
            self._extend_pe(seq_len)

        return x + self.pe[:, :seq_len, :].to(x.dtype)


class TransformerBlock(nn.Module):
    def __init__(
        self           ,
        d_model        ,
        nhead          ,
        dim_feedforward,
        dropout=0.1
    ):
        super().__init__()

        self.self_attn = nn.MultiheadAttention(d_model,
                                               nhead  ,
                                               dropout    =dropout,
                                               batch_first=True   )
        self.norm1    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout  (dropout)

        self.ffn = nn.Sequential(
            nn.Linear (d_model, dim_feedforward),
            nn.ReLU   (),
            nn.Dropout(dropout),
            nn.Linear (dim_feedforward, d_model),
        )
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout  (dropout)

    def forward(self, x, key_padding_mask=None, attn_mask=None):
        attn_output, _ = self.self_attn(x,
                                        x,
                                        x,
                                        key_padding_mask=key_padding_mask,
                                        attn_mask       =attn_mask       )
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)

        ffn_out = self.ffn(x)
        x       = x + self.dropout2(ffn_out)
        x       = self.norm2(x)

        return x


class Transformer_CRF_Tagger(nn.Module):
    def __init__(
        self         ,
        vocab_size   ,
        embedding_dim,
        hidden_dim   ,
        num_labels   ,
        pad_idx      ,

        num_layers     =5,
        num_heads      =8,
        dim_feedforward=None,

        dropout=0.1,
        max_len=512,
    ):
        super().__init__()

        self.num_labels = num_labels
        self.pad_idx    = pad_idx
        self.d_model    = embedding_dim
        if dim_feedforward is None:
            dim_feedforward = embedding_dim * 4

        self.embedding   = nn.Embedding      (vocab_size   ,
                                              embedding_dim,
                                              padding_idx=pad_idx)
        self.pos_enc     = PositionalEncoding(embedding_dim, max_len=max_len)
        self.emb_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList(
            [TransformerBlock(embedding_dim  ,
                              num_heads      ,
                              dim_feedforward,
                              dropout)
             for _ in range(num_layers)]
        )

        self.fc  = nn.Linear(embedding_dim, num_labels)
        self.crf = CRF      (num_labels   , batch_first=True)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding.weight,
                        mean=0.0 ,
                        std =0.02)
        if self.embedding.padding_idx is not None:
            with torch.no_grad():
                self.embedding.weight[self.embedding.padding_idx].fill_(0)

        nn.init.xavier_uniform_(self.fc.weight)
        if self.fc.bias is not None:
            nn.init.constant_(self.fc.bias, 0.0)

    def forward(self, input_ids, lengths, labels=None):
        device = input_ids.device
        mask             = (input_ids != self.pad_idx)
        key_padding_mask = (input_ids == self.pad_idx)

        x = self.embedding  (input_ids)
        x = self.pos_enc    (x)
        x = self.emb_dropout(x)

        for layer in self.layers:
            x = layer(x, key_padding_mask=key_padding_mask)

        emissions = self.fc(x)

        if labels is not None:
            return -self.crf(emissions, labels, mask=mask, reduction="mean")

        return self.crf.decode(emissions, mask=mask)

In [ ]:
transformers_emb = 64

model_transformers = Transformer_CRF_Tagger(
    vocab_size     = vocab_size      ,
    embedding_dim  = transformers_emb,
    hidden_dim     = hidden_dim,
    num_labels     = num_labels,
    pad_idx        = word2idx[PAD_TOKEN],

    num_layers     =2,
    num_heads      =4,
    dim_feedforward=embedding_dim,

    dropout=0.1,
    max_len=256,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW  (model_transformers.parameters(), lr=5e-4)

In [ ]:
total, trainable = count_parameters(model_transformers)

print(f"Total parameters       : {total    :,}")
print(f"Trainable parameters   : {trainable:,}")

In [ ]:
best_val_loss     = float("inf")
epochs_no_improve = 0

num_epochs = 50
patience   =  5

os.makedirs("models", exist_ok=True)

for epoch in range(1, num_epochs + 1):
    model_transformers.train()

    total_loss = 0.0
    for input_ids, labels, lengths in train_loader:
        input_ids = input_ids.to(device)
        labels    = labels   .to(device)
        lengths   = lengths  .to(device)

        optimizer.zero_grad()

        loss = model_transformers(input_ids,
                                  lengths  ,
                                  labels=labels)

        loss .backward()
        optimizer.step()
        total_loss += loss.item()

    model_transformers.eval()

    val_loss = 0.0
    with torch.no_grad():
        for input_ids, labels, lengths in val_loader:
            input_ids = input_ids.to(device)
            labels    = labels   .to(device)
            lengths   = lengths  .to(device)

            val_loss += model_transformers(input_ids,
                                           lengths  ,
                                           labels=labels).item()

    avg_loss     = total_loss / len(train_loader)
    avg_val_loss = val_loss   / len(val_loader  )

    print()
    print(f"Epoch {epoch}/{num_epochs}")
    print(f"  Train Loss : {avg_loss    :.4f}")
    print(f"  Val Loss   : {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss     = avg_val_loss
        epochs_no_improve = 0

        best_model_wts = deepcopy(model_transformers.state_dict())
        torch.save(
            best_model_wts,
            os.path.join("models", "model_transformers.pt")
        )

        print(f"  🔹 Best model saved ({epoch} epoch)!")
    else:
        epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print()
            print(f"⚠️ Early stopping triggered! No improvement for {patience} epochs.")
            break

    model_transformers.eval()
    with torch.no_grad():
        docs, options = displacy_from_predict(model_transformers, tokens, predict_ner_crf)
        displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
transformers_model_path = os.path.join("models", "model_transformers.pt")

model_transformers.load_state_dict(
    torch.load(
        transformers_model_path,
        map_location=device
    )
)
model_transformers.to  (device)
model_transformers.eval()

test_metrics = evaluate_model_crf(model_transformers, test_loader)

print("=== Test Set Results ===")
print(f"  F1 Score   : {test_metrics['overall_f1'       ]:.4f}")
print(f"  Precision  : {test_metrics['overall_precision']:.4f}")
print(f"  Recall     : {test_metrics['overall_recall'   ]:.4f}")
print(f"  Accuracy   : {test_metrics['overall_accuracy' ]:.4f}")

In [ ]:
test_texts = [
    "João encontrou Maria ontem à noite."     ,
    "Estou indo para João Pessoa amanhã cedo.",
    "A Google lançou um novo modelo de IA."   ,
    "A Universidade Federal da Paraíba convidou Ana Beatriz para apresentar sua pesquisa em São Paulo."                 ,
    "O presidente da Microsoft Brasil, André Oliveira, visitou o escritório em Fortaleza para anunciar novas parcerias.",
    "Mariana trabalhou três anos na IBM, antes de se mudar para o Rio de Janeiro para atuar no BNDES."                  ,
    "Em 2024, Carlos Eduardo foi contratado pelo Banco do Brasil após concluir seu mestrado na USP, em São Paulo."      ,
    "A Meta divulgou um relatório em que Sheryl Sandberg mencionou iniciativas de segurança digital nas operações da empresa."                ,
    "Durante a reunião em Brasília, representantes da ONU e do Ministério da Justiça discutiram estratégias para reduzir crimes cibernéticos.",
    "Pedro Henrique trabalhou por cinco anos na Petrobras no Rio de Janeiro, até receber uma proposta da Amazon em Seattle."                  ,
]

nlp = spacy.blank("pt")

for text in test_texts:
    docs, options = displacy_from_predict(model_transformers         ,
                                          [t.text for t in nlp(text)],
                                          predict_ner_crf)
    displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
del model_transformers

torch.cuda.empty_cache()